In [0]:
# 1. Imports
from pyspark.sql.functions import *
from delta.tables import *

In [0]:
# 2. Paths & Table Names
bronze_table = "accenture.manishgautam.bronze_table"
silver_table = "accenture.manishgautam.silver_table"

In [0]:
# 3. Read Data from Bronze (Batch Mode)
# Yahan koi checkpoint nahi chahiye
df_bronze = spark.read.table(bronze_table)

In [0]:
df_bronze.limit(2).display()

In [0]:
df_silver = (
    df_bronze

    # 1. Remove duplicates & nulls
    .dropDuplicates(["TransactionID"])
    .dropna(subset=["TransactionID", "PatientID", "Amount"])

    # 2. Fix data types
    .withColumn("VisitDate",    to_date("VisitDate",    "M/d/yyyy"))
    .withColumn("ServiceDate",  to_date("ServiceDate",  "M/d/yyyy"))
    .withColumn("PaidDate",     to_date("PaidDate",     "M/d/yyyy"))
    .withColumn("Amount",       round(col("Amount"),    2))
    .withColumn("PaidAmount",   round(col("PaidAmount"),2))

    # 4. New Columns
    .withColumn("processing_days",  datediff(col("PaidDate"), col("ServiceDate")))
    .withColumn("pending_amount",   round(col("Amount") - col("PaidAmount"), 2))
    .withColumn("payment_percentage",      round((col("PaidAmount") / col("Amount")) * 100, 2))

    # 5. Payment Status
    .withColumn("payment_status",
        when(col("PaidAmount") >= col("Amount"),  "FULLY_PAID")
        .when(col("PaidAmount") == 0,             "UNPAID")
        .otherwise(                               "PARTIALLY_PAID"))

    # 9. Payor Category
    .withColumn("payor_category",
        when(col("LineOfBusiness") == "MEDICARE",   "GOVERNMENT")
        .when(col("LineOfBusiness") == "MEDICAID",  "GOVERNMENT")
        .when(col("LineOfBusiness") == "COMMERCIAL","PRIVATE")
        .otherwise("OTHER"))

    # 10. Same Day Service Flag
    .withColumn("is_same_day",
        when(col("VisitDate") == col("ServiceDate"), True)
        .otherwise(False))

    # Metadata
    .withColumn("silver_ingestion_time", current_timestamp())

    # 17. Filter invalid records
    .filter(col("Amount") > 0)
    .filter(col("PaidDate") >= col("ServiceDate"))

    # 18. Select & Reorder columns
    .select(
        "TransactionID", "PatientID", "ProviderID", "DeptID",
        "VisitDate", "ServiceDate", "PaidDate", "VisitType",
        "Amount", "PaidAmount", "pending_amount",
        "payment_status", "payment_percentage", "processing_days",
        "payor_category", "is_same_day",
        "ClaimID", "PayorID", "LineOfBusiness",
        "silver_ingestion_time"
    )

    # 19. Cast columns to proper types
    .withColumn("Amount",     col("Amount").cast("double"))
    .withColumn("PaidAmount", col("PaidAmount").cast("double"))
)

# Display result
#df_silver.limit(5).display()

In [0]:
# Drop table if it exists (safer than TRUNCATE for tables that may not exist yet)
#spark.sql(f"DROP TABLE IF EXISTS {silver_table}")

In [0]:

# COMMAND ----------
# 5. Write/Merge Logic (Silver Table)
if not spark.catalog.tableExists(silver_table):
    # First time load: Table create hogi
    (df_silver.write
     .format("delta")
     .mode("overwrite")
     .saveAsTable(silver_table))
    print(f"Success: {silver_table} created and loaded for the first time.")
else:
    # Incremental load: Sirf new/updated data merge hoga
    target_table = DeltaTable.forName(spark, silver_table)
    
    (target_table.alias("target")
     .merge(
         df_silver.alias("updates"),
         "target.id = updates.id" # Join condition
     )
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
    print(f"Success: Data merged into {silver_table}.")


In [0]:
display(spark.table("accenture.manishgautam.silver_table_stream").count())